# WAVEGO Pro HTTP 接口

通过 HTTP 控制和读取机器人，无需网页界面。主程序在 5000 端口提供免认证 API（仅限局域网）。

**安全第一**：运动指令是真实生效的。`T:111` 步态指令为电平式——机器人会一直移动直到收到停止指令。首次实验请将机器人架起，脚本结尾务必停止。

```text
POST /api/cmd   {"T":111,"FB":1,"LR":0}   步态向量 (FB/LR 各 -1|0|+1)
GET  /api/status                            传感器 + 视频 + CV 快照
GET  /api/telemetry                         关节 @1Hz、电池、主机错误
GET/POST /api/cv                            CV 模式切换（人体跟随！）
GET/POST /api/config                        白名单运行时设置
GET  /api/vlm                               VLM 目标会话状态
```

In [ ]:
import requests, json

BASE = "http://127.0.0.1:5000"

def post(path, payload=None, **kw):
    r = requests.post(BASE + path, json=payload, timeout=5, **kw)
    print(r.status_code, r.text[:300])
    return r

def get(path, **kw):
    r = requests.get(BASE + path, timeout=5, **kw)
    print(r.status_code, r.text[:300])
    return r

get("/api/status")

## 步态移动 (T:111)

| FB | LR | 动作 |
|----|----|------|
| 1 | 0 | 前进 |
| -1 | 0 | 后退 |
| 0 | -1 | 左转 |
| 0 | 1 | 右转 |
| 0 | 0 | **停止** |

In [ ]:
import time

post("/api/cmd", {"T": 111, "FB": 0, "LR": 0})   # 先停止
time.sleep(0.5)
post("/api/cmd", {"T": 111, "FB": 1, "LR": 0})   # 前进
time.sleep(1.0)
post("/api/cmd", {"T": 111, "FB": 0, "LR": 0})   # 停止——结尾必须停止

## 同步查询

查询指令（T:106 关节位置、T:207 电池）以负数 `T` 回复。使用 `?sync_ms=` 短轮询等待回复。

In [ ]:
post("/api/cmd?sync_ms=500", {"T": 207})    # -> {"T": -207, "voltage": 7.xx}

## CV 模式与人体跟随

In [ ]:
get("/api/cv")
post("/api/cv", {"mode": "person"})              # 启动跟随（机器人会移动！）
# post("/api/cv", {"mode": "none"})              # 停止跟随
post("/api/cv", {"mode": "none", "motion_lock": True})   # 硬开关：禁止 CV 运动

## 运行时设置（白名单，持久化到 config.yaml）

In [ ]:
get("/api/config")
post("/api/config", {"video.default_quality": 40})

## 遥测快照（关节、电池、主机错误）

In [ ]:
d = get("/api/telemetry").json()
labels = ["LF_A", "LF_B", "LF_H", "LH_A", "LH_B", "LH_H",
          "RF_A", "RF_B", "RF_H", "RH_A", "RH_B", "RH_H"]
fb = (d.get("joints") or {}).get("fb")
if fb:
    for name, v in zip(labels, fb):
        print(f"{name:5s} {v:4d} " + "#" * (v * 40 // 1023))
else:
    print("尚未收到关节帧（主程序内 1Hz 轮询）")
print("电池:", d.get("battery"), "V | 主机错误:", len(d.get("errors", [])))

## VLM 目标会话（语音 / 网页 / API 触发）

In [ ]:
get("/api/vlm")
# post("/api/vlm", {"goal": "找到一个球并停在它前面"})   # 会真实移动！
# post("/api/vlm/stop", {})                              # 急停